# 00c Realistic FG/RM Generation v2

Generates statistically controlled synthetic datasets with FG→RM lag linkage:

- `p_v2_portable_monthly.csv` (FG-only)
- `w_v2_wms_monthly.csv` (FG + operational features)
- `p_v2_rm_monthly.csv` (RM derived demand)
- realism + lag validation reports


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling')
GENERATED = ROOT / 'outputs' / 'generated'
REPORTS = ROOT / 'outputs' / 'reports'
for p in [GENERATED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

fg_master = pd.read_csv(GENERATED / 'fg_rm_foundation_fg_master.csv')
rm_master = pd.read_csv(GENERATED / 'fg_rm_foundation_rm_master.csv')
bom = pd.read_csv(GENERATED / 'fg_rm_foundation_bom.csv')
lead_priors = pd.read_csv(GENERATED / 'fg_rm_foundation_leadtime_priors.csv')

print('FG count:', len(fg_master), 'RM count:', len(rm_master), 'BOM rows:', len(bom))

FG count: 96 RM count: 288 BOM rows: 384


In [2]:
months = pd.date_range('2022-01-01', periods=48, freq='MS')

base_scale = {
    'SOAP': 2200,
    'FACEWASH': 1400,
    'SHAMPOO': 1700,
    'CREAM': 900,
}
cv_map = {
    'SOAP': 0.18,
    'FACEWASH': 0.22,
    'SHAMPOO': 0.20,
    'CREAM': 0.25,
}
promo_rate = {
    'SOAP': 0.12,
    'FACEWASH': 0.16,
    'SHAMPOO': 0.14,
    'CREAM': 0.10,
}

def seasonal_multiplier(month_num: int, fg_category: str) -> float:
    base = 1.0 + 0.16 * np.sin((month_num - 1) * 2 * np.pi / 12.0)
    # Sri Lanka practical peaks: April (new-year season), Nov/Dec festive
    if month_num in [4]:
        base *= 1.16
    if month_num in [11, 12]:
        base *= 1.18
    if fg_category == 'SOAP' and month_num in [6, 7]:
        base *= 1.04
    if fg_category == 'FACEWASH' and month_num in [3, 4, 8]:
        base *= 1.05
    return float(base)

fg_rows = []
for fg in fg_master.itertuples(index=False):
    cat = fg.fg_category
    base = base_scale.get(cat, 1000) * float(rng.uniform(0.6, 1.5))
    trend = float(rng.uniform(-0.03, 0.05))  # annualized drift
    ar = 0.45
    eps_prev = 0.0

    on_hand = base * float(rng.uniform(0.7, 1.4))
    for i, m in enumerate(months):
        mon = int(m.month)
        season = seasonal_multiplier(mon, cat)
        level = base * ((1 + trend) ** (i / 12.0))
        promo = int(rng.random() < promo_rate.get(cat, 0.12))
        holiday = int(mon in [4, 11, 12])
        promo_lift = 1.0 + (rng.uniform(0.08, 0.22) if promo else 0.0)
        holiday_lift = 1.0 + (rng.uniform(0.03, 0.10) if holiday else 0.0)

        cv = cv_map.get(cat, 0.3)
        eps = ar * eps_prev + float(rng.normal(0, cv))
        eps_prev = eps
        latent = max(level * season * promo_lift * holiday_lift * (1.0 + eps), 0.0)

        # inventory dynamics + stockout censoring
        target_cover_months = 1.4
        inbound = max(level * float(rng.uniform(0.9, 1.25)), 0.0)
        available = max(on_hand + inbound, 0.0)
        observed = min(latent, available)
        stockout_gap = max(latent - observed, 0.0)
        stockout_days = int(np.clip(np.round((stockout_gap / max(latent, 1.0)) * 18), 0, 18))
        on_hand = max(available - observed, 0.0)

        supplier_otif = float(np.clip(rng.beta(20, 2.8), 0.55, 0.995))
        lead_time_days = float(np.clip(rng.normal(30, 6), 7, 90))
        discount = float(np.round(rng.uniform(0.05, 0.25), 4) if promo else np.round(rng.uniform(0.0, 0.03), 4))
        returns_qty = int(np.round(observed * rng.uniform(0.003, 0.02)))
        open_so = int(np.round(observed * rng.uniform(0.03, 0.14) + stockout_days * 2.0))

        fg_rows.append({
            'month': m,
            'fg_code': fg.fg_code,
            'fg_name': fg.fg_name,
            'fg_category': fg.fg_category,
            'demand_units': int(np.round(observed)),
            'latent_demand_units': float(latent),
            'promotion_flag': promo,
            'holiday_flag': holiday,
            'price_or_discount': discount,
            'lead_time_days': round(lead_time_days, 2),
            'supplier_otif': round(supplier_otif, 4),
            'inbound_po_qty': int(np.round(inbound)),
            'on_hand_inventory': int(np.round(on_hand)),
            'stockout_days': stockout_days,
            'open_sales_orders': open_so,
            'returns_qty': returns_qty,
        })

fg_df = pd.DataFrame(fg_rows)
fg_df.head()

,month,fg_code,fg_name,fg_category,demand_units,latent_demand_units,promotion_flag,holiday_flag,price_or_discount,lead_time_days,supplier_otif,inbound_po_qty,on_hand_inventory,stockout_days,open_sales_orders,returns_qty
0,2022-01-01,SOAP_001,Bar Soap 001,SOAP,1851,1850.697496,0,0,0.0278,35.28,0.8924,3541,5402,0,223,26
1,2022-02-01,SOAP_001,Bar Soap 001,SOAP,2118,2118.194913,0,0,0.0291,25.91,0.8680,3122,6406,0,245,39
2,2022-03-01,SOAP_001,Bar Soap 001,SOAP,3104,3104.485864,0,0,0.0057,33.70,0.9250,2613,5914,0,256,16
3,2022-04-01,SOAP_001,Bar Soap 001,SOAP,4609,4608.677283,0,1,0.0086,30.70,0.9052,3403,4708,0,209,67
4,2022-05-01,SOAP_001,Bar Soap 001,SOAP,3459,3459.047082,0,0,0.0042,26.17,0.9188,3359,4608,0,358,17


In [3]:
# RM demand derived from FG demand via BOM and lead-lag
lead_prior_map = lead_priors.set_index('rm_family')['lead_time_p50'].to_dict()
rm_name_map = rm_master.set_index('rm_code')['rm_name'].to_dict()
rm_family_map = rm_master.set_index('rm_code')['rm_family'].to_dict()

tmp = fg_df[['month', 'fg_code', 'demand_units']].merge(
    bom[['fg_code', 'rm_code', 'rm_family', 'bom_coef']], on='fg_code', how='inner'
)
tmp['lead_days'] = tmp['rm_family'].map(lead_prior_map).fillna(30)
tmp['lead_months'] = np.clip(np.ceil(tmp['lead_days'] / 30.0), 1, 4).astype(int)
tmp['rm_month'] = tmp['month'] - pd.to_timedelta(tmp['lead_months'] * 30, unit='D')
tmp['rm_month'] = pd.to_datetime(tmp['rm_month']).dt.to_period('M').dt.to_timestamp()
tmp['rm_required_units'] = tmp['demand_units'] * tmp['bom_coef']

rm_df = (
    tmp.groupby(['rm_month', 'rm_code'], as_index=False)['rm_required_units']
    .sum()
    .rename(columns={'rm_month': 'month'})
)
rm_df['fg_code'] = rm_df['rm_code'].map(lambda x: f'RM_{int(x):06d}')
rm_df['fg_name'] = rm_df['rm_code'].map(rm_name_map)
rm_df['fg_category'] = rm_df['rm_code'].map(rm_family_map).fillna('GENERAL')

# procurement noise / constraints
rm_noise = rng.lognormal(mean=0.0, sigma=0.14, size=len(rm_df))
rm_df['demand_units'] = np.maximum(np.round(rm_df['rm_required_units'] * rm_noise), 0).astype(int)
rm_df = rm_df[['month', 'fg_code', 'fg_name', 'fg_category', 'demand_units']].sort_values(['fg_code', 'month'])
rm_df.head()

,month,fg_code,fg_name,fg_category,demand_units
0,2021-11-01,RM_100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,2576
78,2021-12-01,RM_100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,4203
156,2022-01-01,RM_100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,2816
234,2022-03-01,RM_100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,2697
312,2022-04-01,RM_100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,2190


In [4]:
# Outputs: P_v2 (portable), W_v2 (wms features), RM monthly
p_v2 = fg_df[['month', 'fg_code', 'fg_name', 'fg_category', 'demand_units']].copy()
w_v2 = fg_df[[
    'month', 'fg_code', 'fg_name', 'fg_category', 'demand_units',
    'on_hand_inventory', 'stockout_days', 'promotion_flag', 'price_or_discount',
    'lead_time_days', 'supplier_otif', 'inbound_po_qty', 'open_sales_orders',
    'returns_qty', 'holiday_flag'
]].copy()

p_v2_path = GENERATED / 'p_v2_portable_monthly.csv'
w_v2_path = GENERATED / 'w_v2_wms_monthly.csv'
rm_v2_path = GENERATED / 'p_v2_rm_monthly.csv'

p_v2.to_csv(p_v2_path, index=False)
w_v2.to_csv(w_v2_path, index=False)
rm_df.to_csv(rm_v2_path, index=False)

print('Saved:', p_v2_path)
print('Saved:', w_v2_path)
print('Saved:', rm_v2_path)

In [5]:
# Realism report
def seasonal_strength(df):
    out = []
    for sid, g in df.groupby('fg_code'):
        if g['month'].nunique() < 24:
            continue
        s = g.sort_values('month')['demand_units'].to_numpy(dtype=float)
        if len(s) <= 12:
            continue
        ac = np.corrcoef(s[12:], s[:-12])[0, 1]
        if np.isnan(ac):
            continue
        out.append(ac)
    return float(np.mean(out)) if out else np.nan

realism = pd.DataFrame([
    {
        'dataset': 'P_V2',
        'rows': len(p_v2),
        'n_series': p_v2['fg_code'].nunique(),
        'n_months': p_v2['month'].nunique(),
        'demand_mean': p_v2['demand_units'].mean(),
        'demand_std': p_v2['demand_units'].std(),
        'zero_rate': (p_v2['demand_units'] <= 0).mean(),
        'seasonal_strength_lag12': seasonal_strength(p_v2),
    },
    {
        'dataset': 'RM_V2',
        'rows': len(rm_df),
        'n_series': rm_df['fg_code'].nunique(),
        'n_months': rm_df['month'].nunique(),
        'demand_mean': rm_df['demand_units'].mean(),
        'demand_std': rm_df['demand_units'].std(),
        'zero_rate': (rm_df['demand_units'] <= 0).mean(),
        'seasonal_strength_lag12': seasonal_strength(rm_df),
    },
])

realism_path = REPORTS / 'data_realism_report.csv'
realism.to_csv(realism_path, index=False)
realism

,dataset,rows,n_series,n_months,demand_mean,demand_std,zero_rate,seasonal_strength_lag12
0,P_V2,4608,96,48,1733.309679,782.722907,0.0,0.186448
1,RM_V2,3510,78,45,2303.731054,2287.568945,0.0,0.070724


In [6]:
# FG-RM lag validation
fg_tot = p_v2.groupby('month', as_index=False)['demand_units'].sum().rename(columns={'demand_units': 'fg_total'})
rm_tot = rm_df.groupby('month', as_index=False)['demand_units'].sum().rename(columns={'demand_units': 'rm_total'})
j = fg_tot.merge(rm_tot, on='month', how='inner').sort_values('month')

lag_rows = []
for lag in range(-6, 7):
    if lag < 0:
        a = j['fg_total'].iloc[:lag].to_numpy(dtype=float)
        b = j['rm_total'].iloc[-lag:].to_numpy(dtype=float)
    elif lag > 0:
        a = j['fg_total'].iloc[lag:].to_numpy(dtype=float)
        b = j['rm_total'].iloc[:-lag].to_numpy(dtype=float)
    else:
        a = j['fg_total'].to_numpy(dtype=float)
        b = j['rm_total'].to_numpy(dtype=float)
    corr = np.corrcoef(a, b)[0, 1] if len(a) > 5 else np.nan
    lag_rows.append({'lag_months_fg_minus_rm': lag, 'corr': corr})

lag_df = pd.DataFrame(lag_rows).sort_values('lag_months_fg_minus_rm')
lag_path = REPORTS / 'fg_rm_lag_validation.csv'
lag_df.to_csv(lag_path, index=False)
lag_df

,lag_months_fg_minus_rm,corr
0,-6,0.005383
1,-5,-0.171286
2,-4,-0.430711
3,-3,-0.514231
4,-2,-0.374652
5,-1,-0.001980
6,0,0.064891
7,1,0.137172
8,2,0.609560
9,3,0.714442


In [7]:
meta = {
    'seed': 42,
    'period_start': str(months.min().date()),
    'period_end': str(months.max().date()),
    'periods': int(len(months)),
    'fg_series': int(p_v2['fg_code'].nunique()),
    'rm_series': int(rm_df['fg_code'].nunique()),
    'notes': [
        'FG demand generated with category-specific level/cv/promo and AR(1) residual.',
        'RM demand derived from FG using BOM and family lead-time priors.',
        'April and Nov/Dec seasonal effects included for Sri Lanka-like demand peaks.'
    ]
}

meta_path = REPORTS / 'p_v2_generation_metadata.json'
meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('Saved:', realism_path)
print('Saved:', lag_path)
print('Saved:', meta_path)